# 03 -- Softmax Regression on MNIST (Multi-Class Classification)

MNIST handwritten digits, LIBSVM encoding: 784 pixel-intensity features, 10 classes,
60K training / 10K test images.

We fit `models.SoftmaxRegression` -- categorical cross-entropy + L2 regularization
over a **matrix** of weights $W \in \mathbb{R}^{d\times K}$ -- with every optimizer.
Every optimizer in this library operates element-wise, so the exact same
`Adam`/`RMSProp`/... classes used for the binary logistic-regression vector case
work unmodified on the $(d, K)$ softmax weight matrix.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
%matplotlib inline

from datasets.loaders import load_mnist
from models import SoftmaxRegression
from optimizers import build_optimizer
from experiments.compare_optimizers import compare_optimizers
from visualization.plots import (
    plot_loss_curves, plot_accuracy_curves, plot_gradient_norms,
    plot_comparison_bars, plot_confusion_matrix, plot_runtime_vs_loss,
)

ds = load_mnist()
print(ds)

## Loss and gradient

$$\mathcal L(W) = -\frac1n\sum_i \log \frac{e^{(Wx_i)_{y_i}}}{\sum_k e^{(Wx_i)_k}} + \frac{\lambda}{2}\lVert W\rVert_F^2, \qquad \nabla_W \mathcal L = \frac1n X^\top(P - Y_{\text{onehot}}) + \lambda W$$

computed with the log-sum-exp trick for numerical stability.

In [ ]:
optimizer_lrs = {
    'Gradient Descent': 0.5,
    'SGD': 0.1,
    'Adagrad': 0.5,
    'AdaGrad-Norm': 1.0,
    'RMSProp': 0.01,
    'Adam': 0.01,
    'L-BFGS': 1.0,
}
optimizers = {name: build_optimizer(name, lr=lr) for name, lr in optimizer_lrs.items()}

result = compare_optimizers(
    lambda: SoftmaxRegression(n_classes=ds.n_classes, l2=1e-4),
    optimizers,
    ds.X_train, ds.y_train, ds.X_test, ds.y_test,
    epochs=25, batch_size=256, track_memory=True,
)
result.table

In [ ]:
plot_loss_curves(result.histories); plt.show()
plot_accuracy_curves(result.histories); plt.show()
plot_gradient_norms(result.histories); plt.show()
plot_runtime_vs_loss(result.histories); plt.show()

In [ ]:
best_name = result.table['test_accuracy'].astype(float).idxmax()
best_result = result.results[best_name]
print('Best optimizer:', best_name)

model = SoftmaxRegression(n_classes=ds.n_classes, l2=1e-4)
Xb_test = model._add_intercept(ds.X_test)
preds = model.predict(best_result.final_params, Xb_test)
plot_confusion_matrix(ds.y_test, preds, class_names=[str(i) for i in range(10)], title=f'MNIST -- {best_name}')
plt.show()

Continue to **`04_full_comparison_report.ipynb`** for the consolidated cross-dataset report.